# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [ ]:
# Standard
import pandas as pd
import re

# Third-party
import numpy as np
import plotly.express as px
from sklearn.metrics import explained_variance_score, mean_squared_error, mean_absolute_error, r2_score

# Local
from Dataset.ml1m_data_loader import load_and_merge_data, ml_test_train_split
from models.matrix_factorisation.NMF_matrix_factorisation import NMFMatrixFactorisation

### Load dataset

In [ ]:
df = load_and_merge_data()
df_train, df_test = ml_test_train_split(df)

In [ ]:
df_train.head()

### Matrix Factorisation
Matrix factorisation algorithm applied to Top-N and Similarity (by movie) slates.

i.e. answers the questions: "what are the top N movies for a specific user" and "because someone watched a movie, they should watch"


Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [ ]:
NMF_model = NMFMatrixFactorisation(df_train, n_components=50)

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

In [ ]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

In [ ]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

## Top-N movies for user

In [ ]:
user_id  = 44
NMF_model.understand_user_profile(user_id)
rec = NMF_model.user_top_N(user_id)
print(f"Recommendations for user 44:")
display(NMF_model.get_recommend_dataframe(rec))

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [ ]:
def evaluation_metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R Squared": r2_score(y_true, y_pred),
        "Explained variance": explained_variance_score(y_true, y_pred)
    }

In [ ]:
[print(f"{k}: {v:.2f}") for k, v in evaluation_metrics(NMF_model.pivot, NMF_model.V).items()]

## User profile evaluation

In [ ]:
user_ids = [6013, 2195, 1198, 3662, 4713]

In [ ]:
for user_id in user_ids:
    NMF_model.understand_user_profile(user_id, rating_dist=False, wc=False)
    rec = NMF_model.user_top_N(user_id)
    print(f"Recommendations for user {user_id}")
    display(NMF_model.get_recommend_dataframe(rec))